# II) Taylor test

## 1) Principle
The aim is to check the sensitivity of the losses $f$ with respect to a local change of electric conductivity.
The test relies on Taylor expansion
$$f(\sigma + h \delta \sigma) = f(\sigma) + f'(\sigma + \delta \sigma) h + O(|h|^2)  $$
so if the value of $f'$ is correct, the norm of Taylor remainder $|\mathcal T|$ should decrease quadratically with $|h|$
$$|\mathcal T| = f(\sigma) - f'(\sigma + \delta \sigma) h = O(|h|^2)  $$

___
## 2) Application to Joule losses

### 2.a) Setup

#### 2.a.i) Geometry
No need for a very refined mesh! The derivative expression should be independant of the mesh discretization.

In [ ]:
from ngsolve.webgui import Draw
from utils.geometry import machine_mesh

polePairs = 6
bundles_per_half_slot = 5
mesh = machine_mesh(p=polePairs, 
                    bundles_per_half_slot=bundles_per_half_slot,
                    hAirgap = 2e-3)
print(f"{mesh.nv} nodes, {mesh.ne} elements")
Draw(mesh)

#### 2.a.ii) Material properties

In [ ]:
from utils.physics import magnetization_halbach
from ngsolve import pi

mu0 = 4e-7*pi 
mur_iron = 1000
br = 1
sigma_copper = 6e7

M = mesh.MaterialCF({"rotor" : magnetization_halbach(br = br)})
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0* mur_iron)}, default = 1/mu0)
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})

#### 2.a.iii) Current supply

In [ ]:
from utils.supply import phase_current, winding_arrangement, bundle_arrangement

freq = 1000
Irms = 100
phi = 0
winding_type =  "distributed" # "concentrated"

phase = phase_current(I_rms=Irms,  load_angle=phi)
winding = winding_arrangement(phase, type =winding_type)
bundles = bundle_arrangement(winding, bundles_per_half_slot=bundles_per_half_slot)

#### 2.a.iv) Finite element space

In [ ]:
curve_order = 2
fem_order = curve_order

from ngsolve import Periodic, H1
fes = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet =  "shaft|out", 
                   complex = True),  [-1]*7 )

print(f"Number of degrees of freedom : {fes.ndof}")

### 2.b) Definition of objective function and derivative

In [ ]:
from utils.physics import solve_magnetoharmonic
from utils.physics import joule_losses

def F_losses(sigma):
       """ Wrapper to return losses from conductivity sigma field """
       result = solve_magnetoharmonic(fes = fes, 
                        frequency = freq,
                        reluctivity = nu,
                        magnetization = M,
                        conductivity=sigma,
                        supply = bundles)
       return joule_losses(result)   

from utils.optimization import d_joule_losses

def dF_losses(sigma):
       """ Wrapper to return gradient of losses from conductivity sigma field """
       result = solve_magnetoharmonic(fes = fes, 
                        frequency = freq,
                        reluctivity = nu,
                        magnetization = M,
                        conductivity=sigma,
                        supply = bundles)
       return d_joule_losses(result, sigma, wrt = "conductivity")

### 2.c) Taylor tests

In [ ]:
from utils.optimization import taylor_test
from ngsolve import GridFunction, L2

# define the reference conductivity and its regularity (through its function space)
sigma_ref = GridFunction(L2(mesh, definedon = "slot(.*)_bundle.*"))  # L2 means sigma has no continuity
# sigma_ref = GridFunction(H1(mesh, definedon = "slot(.*)_bundle.*"))   # H1 ensures the continuity of sigma

sigma_ref.Set(sigma)

H, taylor_remainder, dF0, dF_estimated = taylor_test(F_losses,  dF_losses,  sigma_ref)


We then plot the norm of Taylor remainder and check its quadratic convergence, until floating point computation errors appear.

In [ ]:
# Display convergence of Taylor remainder
from utils.optimization import plot_taylor_test
import matplotlib.pyplot as plt
plot_taylor_test(H, taylor_remainder); plt.grid()
plt.title(f"Taylor test (sigma space : {sigma_ref.space.type[0:2]}, order = {sigma_ref.space.globalorder})")
plt.show()

We can also plot the convergence of directional derivative.

In [ ]:
# Display convergence of directional derivative to the value found by adjoint method

plt.axhline(dF0, linestyle='--', label = "Adjoint method")
plt.semilogx(H, dF_estimated,'o', label = "Finite difference")
plt.title("Directional derivative"); plt.legend()
plt.grid(); plt.xlabel("h"); plt.ylabel("$f'(\\sigma)(\\delta\\sigma)$")
plt.show()